# AYA Edge 0.2 — First Kaggle Agent-SFT Baseline

This notebook runs the first bounded AYA Edge agent-SFT experiment. It is deliberately a baseline, not a capability claim. It records training loss, validation loss, checkpoint reload status, structured tool-call formatting, and a held-out-tool probe.

The attached Kaggle Dataset must contain the already-materialized, tokenizer-compatible files `train.jsonl`, `validation.jsonl`, `test.jsonl`, and `manifest.json` from `datasets/ultradata_agent_sft_v1/`. The raw source-derived datasets are not downloaded by this notebook.

## Baseline interpretation

- 29 validation examples are sufficient for an initial smoke test, not a reliable generalization assessment.
- 100 steps establish a reproducible baseline only; they do not establish that AYA is a capable agent.
- The materialized subset contains no naturally unseen tool names in validation, so the notebook reports that limitation and additionally probes a synthetic tool schema that is absent from training.
- Tool schemas are validated during data preparation and preflight, but are not repeated in each first-run prompt because the complete trajectories require the 2,048-token context.

In [ ]:
# User-tunable Kaggle knobs. Keep this first baseline bounded.
REPO_URL = 'https://github.com/hygienhub03-dev/Aya-Edge-0.1.git'
REVISION = 'd25cfea'
INPUT_DIR = '/kaggle/input/aya-edge-agent-sft-v1'
OUTPUT_DIR = '/kaggle/working/aya-edge-agent-sft-baseline'
MAX_STEPS = 100
VALIDATION_INTERVAL = 10
BATCH_SIZE = 1
RESUME_CHECKPOINT = None  # Set to an attached .pt file only for a deliberate resume.
assert MAX_STEPS == 100, 'Keep the first baseline at 100 steps until reviewed.'
print({'revision': REVISION, 'max_steps': MAX_STEPS, 'validation_interval': VALIDATION_INTERVAL})

In [ ]:
import hashlib, json, os, pathlib, platform, subprocess, sys, time
from pathlib import Path
import torch
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
WORK = Path('/kaggle/working')
REPO = WORK / 'Aya-Edge-0.1'
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVISION], check=True)
os.chdir(REPO)
HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if not HEAD.startswith(REVISION): raise RuntimeError(f'wrong repository revision: {HEAD}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data]'], check=True)
print({'repo': str(REPO), 'head': HEAD, 'cwd': str(Path.cwd())})

In [ ]:
# Run the repository tests before any CUDA/model work.
tests = subprocess.run([sys.executable, '-m', 'pytest', '-q'], text=True, capture_output=True)
print(tests.stdout)
if tests.stderr: print(tests.stderr[-12000:])
if tests.returncode != 0: raise RuntimeError('Repository tests failed; do not train.')

In [ ]:
# Attach the exact materialized SFT subset.
input_dir = Path(INPUT_DIR)
out_dir = Path(OUTPUT_DIR)
tokenizer_path = REPO / 'artifacts/tokenizer/aya_edge_bpe_32768.json'
tokenizer_manifest = REPO / 'artifacts/tokenizer/aya_edge_bpe_32768.manifest.json'
dataset_manifest = input_dir / 'manifest.json'
train_jsonl = input_dir / 'train.jsonl'
validation_jsonl = input_dir / 'validation.jsonl'
required = [tokenizer_path, tokenizer_manifest, dataset_manifest, train_jsonl, validation_jsonl]
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Attach the prepared Kaggle Dataset; missing: ' + ', '.join(missing))
dataset_data = json.loads(dataset_manifest.read_text())
if dataset_data.get('max_length') != 2048 or dataset_data.get('include_tool_schemas') is not False:
    raise ValueError('Dataset manifest is not the preflighted 2,048-token schema-omitted subset.')
print(json.dumps({'dataset': dataset_data['splits'], 'rejections': dataset_data['rejections']}, indent=2))

In [ ]:
# Authoritative tokenizer/model/data preflight.
preflight = subprocess.run([
    sys.executable, 'scripts/kaggle_preflight.py',
    '--config', 'configs/aya_edge_0_2_language.yaml',
    '--tokenizer', str(tokenizer_path),
    '--train-jsonl', str(train_jsonl),
    '--validation-jsonl', str(validation_jsonl),
    '--manifest', str(dataset_manifest),
    '--output', str(out_dir / 'preflight.json'),
], text=True, capture_output=True)
print(preflight.stdout)
if preflight.stderr: print(preflight.stderr[-12000:])
if preflight.returncode != 0: raise RuntimeError('Kaggle preflight failed; do not train.')
preflight_report = json.loads((out_dir / 'preflight.json').read_text())
assert preflight_report['status'] == 'ready'
assert preflight_report['include_tool_schemas'] is False
print('Preflight ready:', preflight_report['train_examples'], 'train /', preflight_report['validation_examples'], 'validation')

## Tool-call formatting and mask checks

This cell checks that structured assistant tool calls are represented with AYA special markers, that their JSON is part of the assistant target, and that non-assistant context remains masked with `-100`. It also validates the full source trajectories before the context-safe materialized subset is used.

In [ ]:
from aya_edge.tokenizer.bpe import ProductionBPETokenizer, SPECIAL_TOKEN_IDS
from aya_edge.training.agent_sft import encode_trajectory, read_encoded_jsonl
tokenizer = ProductionBPETokenizer.load(tokenizer_path)
encoded_train = read_encoded_jsonl(train_jsonl, tokenizer, max_length=2048, include_tool_schemas=False)
assert encoded_train and all(example.supervised_tokens > 0 for example in encoded_train)
tool_rows = []
for line in train_jsonl.read_text(encoding='utf-8').splitlines():
    row = json.loads(line)
    if any(message.get('tool_calls') for message in row.get('messages', [])):
        tool_rows.append(row)
        if len(tool_rows) >= 3: break
assert tool_rows, 'No structured tool-call rows found in the materialized training subset.'
for row in tool_rows:
    example = encode_trajectory(row, tokenizer, max_length=2048, include_tool_schemas=False)
    decoded = tokenizer.decode(example.input_ids)
    assert '<|tool_call|>' in decoded
    tool_id = SPECIAL_TOKEN_IDS['<|tool_call|>']
    assert tool_id in example.input_ids
    tool_position = example.input_ids.index(tool_id)
    assert example.labels[tool_position] != -100, 'Tool-call marker is not supervised.'
    assert example.labels[0] == -100
print({'checked_tool_call_rows': len(tool_rows), 'assistant_only_masks': True, 'structured_marker': '<|tool_call|>'})

In [ ]:
# Run the exact first baseline through the repository's native agent-SFT loop.
out_dir.mkdir(parents=True, exist_ok=True)
checkpoint = out_dir / 'aya_edge_agent_sft_baseline.pt'
metrics = out_dir / 'metrics.jsonl'
command = [
    sys.executable, 'scripts/train_agent_sft.py',
    '--config', 'configs/aya_edge_0_2_language.yaml',
    '--tokenizer', str(tokenizer_path),
    '--input-jsonl', str(train_jsonl),
    '--validation-jsonl', str(validation_jsonl),
    '--output', str(checkpoint),
    '--metrics', str(metrics),
    '--max-steps', str(MAX_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--validation-interval', str(VALIDATION_INTERVAL),
]
if RESUME_CHECKPOINT: command.extend(['--resume', RESUME_CHECKPOINT])
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
assert checkpoint.is_file() and metrics.is_file()
print('Checkpoint:', checkpoint, 'bytes=', checkpoint.stat().st_size)

In [ ]:
# Reload the saved state into a fresh model and recompute validation loss.
import yaml
from aya_edge import AyaEdgeConfig, AyaEdgeForCausalLM
from aya_edge.training.checkpoint import load_checkpoint
from scripts.train_agent_sft import evaluate
cfg = AyaEdgeConfig(**yaml.safe_load(Path('configs/aya_edge_0_2_language.yaml').read_text())['model'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
reload_model = AyaEdgeForCausalLM(cfg).to(device)
reload_optimizer = torch.optim.AdamW(reload_model.parameters(), lr=1e-4)
reload_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(reload_optimizer, T_max=MAX_STEPS)
reload_scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
metadata = load_checkpoint(checkpoint, reload_model, reload_optimizer, scheduler=reload_scheduler, scaler=reload_scaler, map_location=device)
reloaded_validation_loss, reloaded_tokens = evaluate(reload_model, read_encoded_jsonl(validation_jsonl, tokenizer, max_length=2048, include_tool_schemas=False), batch_size=1, device=device)
assert metadata['step'] == MAX_STEPS
assert reloaded_tokens == preflight_report['validation_supervised_tokens']
print({'checkpoint_reload': 'ok', 'step': metadata['step'], 'validation_loss': reloaded_validation_loss, 'supervised_tokens': reloaded_tokens})

## Generated tool-call inspection

This is a format probe, not a capability benchmark. It checks whether the trained checkpoint emits the expected tool-call marker and whether the following payload can be parsed as JSON.

In [ ]:
from jsonschema import Draft202012Validator
reload_model.eval()
prompt = '<|bos|><|im_start|>system<|im_sep|>You may call tools.<|im_end|><|im_start|>user<|im_sep|>What is the weather in Paris?<|im_end|><|im_start|>assistant<|im_sep|>'
prompt_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
with torch.no_grad():
    generated_ids = reload_model.generate(prompt_ids, max_new_tokens=96, eos_token_id=SPECIAL_TOKEN_IDS['<|eos|>'], temperature=0.7, top_k=40)
generated_text = tokenizer.decode(generated_ids[0].tolist())
print(generated_text)
structured = '<|tool_call|>' in generated_text
json_valid = False
if structured:
    candidate = generated_text.split('<|tool_call|>', 1)[1].split('<|im_end|>', 1)[0].strip()
    try:
        json.loads(candidate)
        json_valid = True
    except json.JSONDecodeError:
        pass
print({'tool_call_marker_emitted': structured, 'tool_call_json_parseable': json_valid, 'interpretation': 'format probe only'})

In [ ]:
# Held-out-tool probe. First check for natural unseen tools; otherwise use a synthetic schema.
def tool_names(path):
    names = set()
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        for tool in row.get('tools') or []:
            fn = tool.get('function', tool)
            if isinstance(fn, dict) and fn.get('name'): names.add(fn['name'])
        for message in row.get('messages', []):
            for call in message.get('tool_calls') or []:
                fn = call.get('function', call)
                if isinstance(fn, dict) and fn.get('name'): names.add(fn['name'])
    return names
train_tool_names = tool_names(train_jsonl)
validation_tool_names = tool_names(validation_jsonl)
natural_unseen = sorted(validation_tool_names - train_tool_names)
held_out_schema = {'type': 'object', 'properties': {'city': {'type': 'string'}, 'currency': {'type': 'string'}}, 'required': ['city'], 'additionalProperties': False}
held_out_tool = 'get_exchange_rate'
probe_prompt = '<|bos|><|im_start|>system<|im_sep|>Use the tool get_exchange_rate with JSON arguments when needed.<|im_end|><|im_start|>user<|im_sep|>What is the exchange rate for Paris today?<|im_end|><|im_start|>assistant<|im_sep|>'
probe_ids = torch.tensor([tokenizer.encode(probe_prompt)], dtype=torch.long, device=device)
with torch.no_grad():
    probe_output = reload_model.generate(probe_ids, max_new_tokens=96, eos_token_id=SPECIAL_TOKEN_IDS['<|eos|>'], temperature=0.7, top_k=40)
probe_text = tokenizer.decode(probe_output[0].tolist())
probe_args_valid = False
if '<|tool_call|>' in probe_text:
    candidate = probe_text.split('<|tool_call|>', 1)[1].split('<|im_end|>', 1)[0].strip()
    try:
        parsed = json.loads(candidate)
        if isinstance(parsed, dict) and 'arguments' in parsed: Draft202012Validator(held_out_schema).validate(parsed['arguments']); probe_args_valid = True
    except Exception:
        pass
print({'natural_unseen_validation_tools': natural_unseen, 'natural_unseen_available': bool(natural_unseen), 'synthetic_tool': held_out_tool, 'synthetic_arguments_schema_valid': probe_args_valid, 'probe_output': probe_text})
if not natural_unseen: print('Caveat: this dataset split has no naturally unseen validation tool names; synthetic probe is not a substitute for a held-out benchmark.')

In [ ]:
# Persist a compact baseline report for review before scaling.
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()
metrics_rows = [json.loads(line) for line in metrics.read_text().splitlines() if line.strip()]
summary = {
    'status': 'AYA Edge 0.2 first Kaggle agent-SFT baseline',
    'git_revision': HEAD,
    'max_steps': MAX_STEPS,
    'validation_interval': VALIDATION_INTERVAL,
    'train_examples': preflight_report['train_examples'],
    'validation_examples': preflight_report['validation_examples'],
    'tokenizer_sha256': sha256(tokenizer_path),
    'dataset_manifest_sha256': sha256(dataset_manifest),
    'checkpoint': str(checkpoint),
    'checkpoint_sha256': sha256(checkpoint),
    'metrics': str(metrics),
    'last_metrics': metrics_rows[-1] if metrics_rows else None,
    'caveat': '29 validation examples and 100 steps are an initial smoke baseline, not a generalization or capability assessment.',
}
(out_dir / 'baseline_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))